## Sanipy — Telco Customer Churn Demonstration

In [ ]:
# Install sanipy from production PyPI
!pip install -q sanipy==0.1.1

## 2. Import Libraries

Import the necessary libraries for data processing and sanity checks.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd

import sanipy
from sanipy import check_dataset, compare_train_test, SanipyConfig

print(f"Sanipy version: {sanipy.__version__}")

## 3. Locate and Load Dataset

We check both Kaggle environment paths and standard local paths for `WA_Fn-UseC_-Telco-Customer-Churn.csv`.

In [ ]:
csv_filename = "WA_Fn-UseC_-Telco-Customer-Churn.csv"
data_path = None

# 1. Search Kaggle input paths recursively if running on Kaggle
kaggle_input = Path("/kaggle/input")
if kaggle_input.exists():
    for path in kaggle_input.glob(f"**/{csv_filename}"):
        data_path = path
        break

# 2. Search local directories
if not data_path:
    candidate_paths = [
        Path(csv_filename),
        Path("data") / csv_filename,
        Path("../data") / csv_filename,
        Path("notebooks") / csv_filename,
    ]
    for path in candidate_paths:
        if path.exists():
            data_path = path
            break

if data_path:
    print(f"Dataset located at: {data_path.resolve()}")
    df = pd.read_csv(data_path)
else:
    print(f"Dataset '{csv_filename}' not found.")
    print("\n--- Setup Instructions ---")
    print("If running on Kaggle: Add 'Telco Customer Churn' dataset from Kaggle's Add Data panel.")
    print("If running locally: Download the dataset and place it in the same directory as this notebook.")
    print("Optionally, install Kaggle CLI and download via command line:")
    print("  kaggle datasets download -d blastchar/telco-customer-churn -p data/ --unzip")
    print("--------------------------")

## 4. Basic Data Inspection

Let's look at the shape, columns, and target distribution of our dataset.

In [ ]:
if 'df' in locals():
    print(f"DataFrame dimensions: {df.shape[0]} rows, {df.shape[1]} columns")
    display(df.head())
    print("\nTarget value counts:")
    print(df["Churn"].value_counts(normalize=True))

## 5. Run Sanipy on Raw Dataset (First Check)

We run Sanipy's primary dataset checker `check_dataset` on the raw, unmodified data. Note that `TotalCharges` is currently a string/object column because it contains blank values.

In [ ]:
if 'df' in locals():
    raw_report = check_dataset(df, target="Churn", task="classification")
    print(raw_report.summary())

## 6. Type Correction: `TotalCharges` to Numeric

As we can observe in the initial run or dataset inspection, `TotalCharges` is treated as a non-numeric feature. Some rows contain empty spaces `" "`, which prevents pandas from automatically parsing it as numeric.

Let's convert `TotalCharges` to numeric, coercing errors to `NaN`, and see how this changes Sanipy's analysis.

In [ ]:
if 'df' in locals():
    df_fixed = df.copy()
    
    # Convert column types and coerce whitespace to NaN
    df_fixed["TotalCharges"] = pd.to_numeric(df_fixed["TotalCharges"], errors="coerce")
    
    print(f"Missing values in 'TotalCharges' after correction: {df_fixed['TotalCharges'].isnull().sum()}")

## 7. Run Sanipy on Fixed Dataset (Second Check)

Now we run `check_dataset` on `df_fixed`. We expect to see that `TotalCharges` is properly recognized as a continuous numeric feature.

In [ ]:
if 'df_fixed' in locals():
    fixed_report = check_dataset(df_fixed, target="Churn", task="classification")
    print(fixed_report.summary())

## 8. Display Sanipy Issues as a DataFrame

To inspect report issues programmatically, we write a helper function to flatten the diagnostics into a pandas DataFrame.

In [ ]:
def issues_to_frame(report):
    rows = []
    for issue in report.issues:
        rows.append({
            "severity": getattr(issue.severity, "value", issue.severity),
            "category": issue.category,
            "column": getattr(issue, "column", None) or (issue.columns[0] if getattr(issue, "columns", None) else None),
            "title": issue.title,
            "message": issue.message,
            "recommendation": issue.recommendation,
        })
    return pd.DataFrame(rows)

if 'fixed_report' in locals():
    df_issues = issues_to_frame(fixed_report)
    display(df_issues)

## 9. Export Reports to JSON and Markdown

Export the diagnostics reports to JSON and Markdown formats for storage, pipelines, or dashboards.

In [ ]:
if 'fixed_report' in locals():
    OUTPUT_DIR = Path("sanipy_telco_outputs")
    OUTPUT_DIR.mkdir(exist_ok=True)
    
    # Export raw dataset report
    raw_report.to_json(OUTPUT_DIR / "raw_dataset_report.json")
    raw_report.to_markdown(OUTPUT_DIR / "raw_dataset_report.md")
    
    # Export fixed dataset report
    fixed_report.to_json(OUTPUT_DIR / "fixed_dataset_report.json")
    fixed_report.to_markdown(OUTPUT_DIR / "fixed_dataset_report.md")
    
    print(f"Saved files to {OUTPUT_DIR.resolve()}:")
    for file_path in OUTPUT_DIR.iterdir():
        print(f" - {file_path.name}")

## 10. Train/Test Split Comparison

In Machine Learning pipelines, it is crucial to ensure train and test splits do not have distribution shifts, unseen categories, or split leakages.

Let's create a train/test split using pandas and compare them using `compare_train_test`.

In [ ]:
if 'df_fixed' in locals():
    # Perform train/test split (80% train, 20% test)
    train_df = df_fixed.sample(frac=0.8, random_state=42)
    test_df = df_fixed.drop(train_df.index)
    
    print(f"Training split: {train_df.shape[0]} rows")
    print(f"Testing split:  {test_df.shape[0]} rows")
    
    # Compare splits
    comparison = compare_train_test(
        train_df,
        test_df,
        target="Churn",
        task="classification",
    )
    print(comparison.summary())

Display the train/test comparison diagnostics as a DataFrame and save the outputs.

In [ ]:
if 'comparison' in locals():
    df_comp = issues_to_frame(comparison)
    display(df_comp)
    
    # Export comparison reports
    comparison.to_json(OUTPUT_DIR / "train_test_comparison.json")
    comparison.to_markdown(OUTPUT_DIR / "train_test_comparison.md")
    
    print("Comparison reports successfully saved to output directory.")

## 11. Run Sanipy CLI Commands inside Jupyter Notebook

Sanipy includes a robust command-line interface. Let's write our DataFrames to temporary CSV files to verify and demonstrate CLI functionalities.

In [ ]:
if 'df_fixed' in locals():
    CLI_DIR = Path("sanipy_telco_cli")
    CLI_DIR.mkdir(exist_ok=True)
    
    full_csv = CLI_DIR / "telco_full.csv"
    train_csv = CLI_DIR / "telco_train.csv"
    test_csv = CLI_DIR / "telco_test.csv"
    
    df_fixed.to_csv(full_csv, index=False)
    train_df.to_csv(train_csv, index=False)
    test_df.to_csv(test_csv, index=False)
    
    print(f"CLI CSV files stored in: {CLI_DIR.resolve()}")

In [ ]:
!sanipy --version

In [ ]:
!python -m sanipy --help

In [ ]:
!sanipy check sanipy_telco_cli/telco_full.csv --target Churn --task classification

In [ ]:
!sanipy compare sanipy_telco_cli/telco_train.csv sanipy_telco_cli/telco_test.csv --target Churn --task classification

In [ ]:
!sanipy check sanipy_telco_cli/telco_full.csv --target Churn --task classification --format json --out sanipy_telco_outputs/cli_dataset_report.json

## 12. Interpretation of Sanipy Warnings

### What Sanipy Found
1. **ID Column (`customerID`)**: Flagged correctly as a high-uniqueness identifier column. In standard model training pipelines, this column should be dropped or used only as an index.
2. **Type Discrepancy (`TotalCharges`)**: Correctly highlighted when `TotalCharges` was loaded as object/string due to space characters. Running Sanipy again after manual conversion confirms clean continuous feature diagnostics.
3. **Target Imbalance**: Warned that the `Churn` target is class-imbalanced (~26% Churn rate). This recommends techniques such as stratified train/test splitting, F1/ROC-AUC evaluation, and class weighting.
4. **Train/Test Comparison**: Evaluated data shift and category overlap consistency between training and testing subsets, ensuring features align correctly.

### Disclaimer & Scope
- **Rule-based heuristics**: Sanipy is built to perform fast, rule-based data sanity checks using lightweight Pandas operations. It does not train ML models or perform complex inference.
- **No Automatic Cleaning**: Sanipy is diagnostic. It detects issues but does not modify your dataset automatically.
- **Human-in-the-loop**: The warnings provided by Sanipy are heuristics. Not every warning implies a critical error; developers must review and interpret diagnostics contextually.

## 13. Optional Cleanup

Run this optional cell to clean up directory files created during the CLI and export steps.

In [ ]:
# import shutil
# if Path("sanipy_telco_outputs").exists(): shutil.rmtree("sanipy_telco_outputs")
# if Path("sanipy_telco_cli").exists(): shutil.rmtree("sanipy_telco_cli")